In [0]:
data = [
    (1001, "Laptop", "North", 2, 800),
    (1002, "Mobile", "South", 3, 400),
    (1003, "Keyboard", "East", 5, 50),
    (1004, "Monitor", "West", 2, 300),
    (1005, "Tablet", None, 1, 300),
    (1002, "Mobile", "South", 3, 400)
]

columns = [
    "order_id",
    "product",
    "region",
    "quantity",

    
    "unit_price"
]


orders_df=spark.createDataFrame(data,columns)


In [0]:
raw_count=orders_df.count()
print("Raw row count :",raw_count)

In [0]:
from pyspark.sql import functions as F
raw_revenue=orders_df.agg(
    F.sum(
        F.col("quantity")*F.col("unit_price")
    ).alias("raw_revenue")
)
display(raw_revenue)





Step C — Identify exact duplicate records

In [0]:
row_counts=orders_df.groupBy(*orders_df.columns).count()
display(row_counts)


In [0]:
exact_duplicates=row_counts.filter(
    F.col("count")>1
)

display(exact_duplicates)

In [0]:
exact_duplicates=(orders_df.groupBy(*orders_df.columns).agg(F.count("*").alias("copies")).filter(F.col("copies")>1))


In [0]:
display(orders_df)

In [0]:
orders_df.groupBy("product").count().show()


In [0]:
orders_df.groupBy("order_id").count().show()


In [0]:
product_count=orders_df.groupBy("product").count()

display(product_count)

### GroupBy and Count

groupBy() groups records based on matching values.

count() calculates the number of records in each group.

Business use:
Identify how frequently products appear in e-commerce orders.

In [0]:
repeated_columns=orders_df.groupBy("product").count()

repeated_products=repeated_columns.filter(
    F.col("count")>1
)
display(repeated_products)

In [0]:
orders_counts=orders_df.groupBy("order_id").count()

duplicate_orders=orders_counts.filter(
    F.col("count")>1
)
display(duplicate_orders)

In [0]:
duplicate_orders.show()

In [0]:
duplicate_orders.select(
    F.sum(F.col("count")-1).alias("extra_records")
).show()

# This is useful because a data engineer should measure the impact of a data-quality issue, not merely identify it.

In [0]:
orders_df.agg(
    F.count("*").alias("total_records"),
    F.countDistinct("order_id").alias("unique_orders")
).show()

In [0]:
# percentage of extra records= ((total_record-distinct_record)/total_record)*100
total_records=orders_df.count()
unique_records=orders_df.select("order_id").distinct().count()
percentage_of_duplicate_records=((total_records-unique_records)/total_records)*100
display(percentage_of_duplicate_records)


### PySpark Concepts Practiced

- `groupBy()`: Groups records by a column.
- `count()`: Counts records in each group.
- `filter()`: Filters groups based on a condition.
- `agg()`: Calculates summary metrics.
- `countDistinct()`: Counts unique values.
- Duplicate percentage: Measures repeated order ID occurrences.

**Learning:** Repeated order IDs do not automatically mean duplicate records.
We need to inspect the actual records before deciding what to remove.

In [0]:
orders_df.groupBy("product").agg(
    F.sum(
        F.col("quantity")*F.col("unit_price")
    ).alias("total_revenue")).show()

In [0]:
orders_df.groupBy("product").agg(
    F.avg("unit_price").alias("average_unit_price")
).show()


In [0]:
orders_df.groupBy("product").agg(
    F.sum("quantity").alias("total_quantity_sold")
).show()

In [0]:
orders_df.groupBy("product").agg(
    F.sum(
        F.col("quantity")*F.col("unit_price")
    ).alias("total_revenue"),
    F.avg("unit_price").alias("average_unit_price"),
    F.sum("quantity").alias("total_quantity_sold")
).show()